# SimHash analyse from Java Stack
## Data
Get the data from the sqlite Database

In [10]:
import sqlite3
import os
import pandas as pd

DATABASE_PATH = os.path.join(os.getcwd(), '..', '..', '03_OpenWPM', 'datadir', "2024-12-crawl-data.sqlite")

DATA_PATH = os.path.join(os.getcwd(), '..', '..', '03_OpenWPM', 'datadir', '2024-12-sources', '2024-12-sources')

def read_from_database(db_path, query):
    """
    Reads data from a SQLite database.

    Args:
        db_path (str): Path to the SQLite database file.
        query (str): SQL query to execute.

    Returns:
        list: Results of the query as a list of tuples.
    """
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Execute the query
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Close the connection
        conn.close()

        return results
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        return None

# SQL query to execute to gather JavaScript code
sql_query = """SELECT sv.site_url, res.url, res.content_hash
                    FROM http_responses res, site_visits sv
                    WHERE sv.visit_id = res.visit_id
                    AND sv.browser_id = res.browser_id
                    AND res.headers LIKE '%content-type","%javascript%'
                    AND res.content_hash NOT NULL
                    """
data = read_from_database(DATABASE_PATH, sql_query)

df = pd.DataFrame(data, columns=["site_url", "url", "content_hash"])

df

,site_url,url,content_hash
0,http://google.com,https://www.google.com/xjs/_/js/k=xjs.hd.de.jU...,30e68528008b0b341858314328ad7b249e40d107ba4c88...
1,http://google.com,https://www.gstatic.com/og/_/js/k=og.qtm.en_US...,0b38ed06a04737d3d9acd1886dd48b2279280c920cf1a3...
2,http://google.com,https://apis.google.com/_/scs/abc-static/_/js/...,2eb562f20da951c71dd407e70133ff3837061234a6d257...
3,http://google.com,https://www.google.com/xjs/_/js/k=xjs.hd.de.jU...,f58baa16706d24713cc270411a7f78344ee414c3204352...
4,http://google.com,https://www.google.com/xjs/_/js/k=xjs.hd.de.jU...,903e94d8b30a915be191081023e7af216b38187b2e3c1f...
...,...,...,...
58500,http://filmyfly.contact,https://imasdk.googleapis.com/js/sdkloader/ima...,71a8a6ad479406d70e8fe167616aade1b6e9fcda252f85...
58501,http://filmyfly.contact,https://s0.2mdn.net/instream/video/client.js,d0bffc7261df1454c5e05475cda7d9e6647318dc6c3936...
58502,http://filmyfly.contact,https://s0.2mdn.net/instream/video/client.js,d0bffc7261df1454c5e05475cda7d9e6647318dc6c3936...
58503,http://filmyfly.contact,https://s0.2mdn.net/instream/video/client.js,d0bffc7261df1454c5e05475cda7d9e6647318dc6c3936...


### Read the files based on the content hash and perform the preproccesing
For the preproccesing, there are some stopwords for JavaScript which will be removed. Those stopwords should be evaluated in a further analysis of overlap.

In [42]:
from glob import glob
import re
from datasketch import MinHash
import random

JS_STOPWORDS = {
    'this', 'function', 'var', 'let', 'const', 'return', 'if', 'else', 'for',
    'while', 'do', 'try', 'catch', 'switch', 'case', 'break', 'continue',
    'typeof', 'new', 'in', 'instanceof', 'true', 'false', 'null', 'undefined'
}

def read_files(content_hash):
    # Should be len == 1
    file = glob(os.path.join(DATA_PATH, content_hash))
    if len(file) == 1:
        minhash = MinHash()
        with open(file[0], 'rb') as f:
            tmp_content = f.read()

            def check_encodings(t):
                # try to decode the text in utf-8 or ascii
                try:
                    return True, t.decode('utf-8')
                except Exception as e:
                    try:
                        return True, t.decode('ascii')
                    except Exception as e:
                        return False, ""

            checked, tmp_content_decoded = check_encodings(tmp_content)

            if checked:
                # Pre processing the data
                # Remove unwanted characters that is NOT a letter, digit
                tmp_content_decoded = re.sub(r'[^a-zA-Z0-9\s]+', ' ', tmp_content_decoded)
                tmp_content_decoded = tmp_content_decoded.lower()
                tokens = tmp_content_decoded.split() # split text on whitespace
                tokens = [tok for tok in tokens if tok not in JS_STOPWORDS]

                for token in tokens:
                    minhash.update(token.encode('utf-8'))

                return minhash

df['minhash'] = df['content_hash'].apply(read_files)
l = df['minhash'].dropna().tolist()
l

KeyboardInterrupt: 

In [39]:
import numpy as np
def build_distance_matrix(data):
    distance_matrix = []

    for i in range(len(data)):
        distances = []
        for j in range(len(data)):
            set_a = data[i]
            set_b = data[j]
            distances.append(set_a.jaccard(set_b))

        distance_matrix.append(distances)

    return distance_matrix

distance_matrix = build_distance_matrix(l)
np.matrix(distance_matrix)

matrix([[1.       , 0.0390625, 0.015625 , ..., 0.0234375, 0.0625   ,
         0.0078125],
        [0.0390625, 1.       , 0.28125  , ..., 0.1328125, 0.0546875,
         0.0234375],
        [0.015625 , 0.28125  , 1.       , ..., 0.078125 , 0.046875 ,
         0.0703125],
        ...,
        [0.0234375, 0.1328125, 0.078125 , ..., 1.       , 0.0546875,
         0.03125  ],
        [0.0625   , 0.0546875, 0.046875 , ..., 0.0546875, 1.       ,
         0.015625 ],
        [0.0078125, 0.0234375, 0.0703125, ..., 0.03125  , 0.015625 ,
         1.       ]])

### DBScan

In [40]:
from sklearn.cluster import DBSCAN
def cluster_info(labels):
    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)

    print("Estimated number of clusters: %d" % n_clusters_)
    print("Estimated number of noise points: %d" % n_noise_)
    print("Cluster labels:",  labels)

clustering_algo = DBSCAN(metric="precomputed")
dbscan_fit = clustering_algo.fit(distance_matrix)
labels_dbscan = dbscan_fit.labels_
cluster_info(labels_dbscan)

Estimated number of clusters: 1
Estimated number of noise points: 0
Cluster labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
from glob import glob
import os
import re
from datasketch import MinHash
import concurrent.futures
from tqdm import tqdm
import pickle

JS_STOPWORDS = {
    'this', 'function', 'var', 'let', 'const', 'return', 'if', 'else', 'for',
    'while', 'do', 'try', 'catch', 'switch', 'case', 'break', 'continue',
    'typeof', 'new', 'in', 'instanceof', 'true', 'false', 'null', 'undefined'
}

def read_file(content_hash):
    file = glob(os.path.join(DATA_PATH, content_hash))
    if len(file) == 1:
        minhash = MinHash()
        with open(file[0], 'rb') as f:
            tmp_content = f.read()

        def check_encodings(t):
            try:
                return True, t.decode('utf-8')
            except Exception:
                try:
                    return True, t.decode('ascii')
                except Exception:
                    return False, ""

        checked, tmp_content_decoded = check_encodings(tmp_content)

        if checked:
            tmp_content_decoded = re.sub(r'[^a-zA-Z0-9\s]+', ' ', tmp_content_decoded)
            tmp_content_decoded = tmp_content_decoded.lower()
            tokens = tmp_content_decoded.split()
            tokens = [tok for tok in tokens if tok not in JS_STOPWORDS]

            for token in tokens:
                minhash.update(token.encode('utf-8'))

            return pickle.dumps(minhash)
    return None

with concurrent.futures.ThreadPoolExecutor(max_workers=50) as executor:
    results = list(tqdm(executor.map(read_file, df['content_hash']), total=len(df)))
    df['minhash'] = results

In [6]:
df.to_csv('simhash.csv', index=False)

In [ ]:
import numpy as np
def build_distance_matrix(data):
    distance_matrix = []

    for i in range(len(data)):
        distances = []
        for j in range(len(data)):
            set_a = data[i]
            set_b = data[j]
            distances.append(set_a.jaccard(set_b))

        distance_matrix.append(distances)

    return distance_matrix

distance_matrix = build_distance_matrix(l)
np.matrix(distance_matrix)

# SimHash2
Using SimHash2 for hashing the content of the file.
Pre processing contains an n_gram of 3, lower case and removes whitespace. Also stopwords will be removed.

In [43]:
import re
from simhash2 import Simhash, SimhashIndex
from tqdm import tqdm
import concurrent.futures
from glob import glob
import pickle

def get_features(s):
    width = 3 # n_gram = 3
    s = s.lower()
    s = re.sub(r'[^\w]+', '', s)
    return [s[i:i + width] for i in range(max(len(s) - width + 1, 1))]

def get_simhash(content_hash):
    file = glob(os.path.join(DATA_PATH, content_hash))
    if len(file) == 1:
        with open(file[0], 'rb') as f:
            tmp_content = f.read()

            def check_encodings(t):
                try:
                    return True, t.decode('utf-8')
                except Exception:
                    try:
                        return True, t.decode('ascii')
                    except Exception:
                        return False, ""

            _, content = check_encodings(tmp_content)

            if _:
                s1 = Simhash(get_features(content))
                #list_of_dups = index.get_near_dups(s1)
                return s1.value

tqdm.pandas()
df['simhash'] = df['content_hash'].progress_apply(get_simhash)

100%|██████████| 58505/58505 [20:15:26<00:00,  1.25s/it]         


In [45]:
df.to_csv('simhash.csv', index=False)

list_of_records = []
for index, row in tqdm(df.iterrows(), total=len(df), desc='Sim Hashing...'):
    site_url = row['site_url']
    url = row['url']
    content_hash = row['content_hash']
    try:
        simhash, list_of_near_dups = get_simhash(content_hash, SimhashIndex(objs, k=3))
        list_of_records.append({'site_url': site_url, 'url':url, 'content_hash':content_hash, 'simhash':simhash, 'near_dups':list_of_near_dups})
    except Exception as e:
        continue